# 04 - Funnel Analysis

## Objective

This notebook analyzes the captain onboarding funnel from signup to approval.

The analysis focuses on two questions from Part A of the assessment:

1. **A1:** What does the signup → approved funnel look like, and where is volume lost at each stage?
2. **A2:** Which onboarding leak is the largest actionable opportunity, and how many additional approved captains could it generate if addressed?

The analysis will use an explicit cohort definition, stage-level denominators, and volume-based drop-off measures. Funnel results will be segmented by relevant captain attributes to identify large, explainable, and actionable differences.


## 1. Load Processed Captain Dataset

The cleaned captain-level dataset produced in Notebook 03 is used as the starting point for funnel analysis.

Using the processed dataset keeps data preparation separate from business analysis and allows this notebook to be reproduced independently.

In [2]:
import pandas as pd
import numpy as np

In [3]:
captain_base = pd.read_csv(
    "../Data/captain_base.csv",
    parse_dates=[
        "signup_ts",
        "decision_ts",
        "first_order_ts"
    ]
)

In [4]:
print("Rows:", len(captain_base))
print("Unique captains:", captain_base["captain_id"].nunique())

Rows: 25000
Unique captains: 25000


## 2. Define the Primary Funnel Cohort

The dataset is extracted at the end of **2026-06-30**, so captains who signed up shortly before the cutoff may not have had enough time to complete onboarding.

To reduce right-censoring, the primary funnel cohort includes captains with at least **14 days of observable time** between signup and the extraction cutoff.

This produces a primary analysis cohort of captains whose onboarding outcomes can be evaluated more fairly.

Captains with less than 14 days of observation are retained in the underlying dataset but excluded from the primary funnel denominator.


In [5]:
funnel = captain_base[
    captain_base["eligible_for_funnel"]
].copy()

In [6]:
print("Total signups:", len(captain_base))
print("Funnel-eligible signups:", len(funnel))
print("Censored signups:", (~captain_base["eligible_for_funnel"]).sum())

Total signups: 25000
Funnel-eligible signups: 22757
Censored signups: 2243


## 3. Define Funnel Stages

The onboarding process is sequential, so funnel stages are defined as **cumulative progression** rather than independent document-clearance rates.

A captain is counted at a stage only if they have successfully cleared that stage and all preceding required stages.

The required sequence is:

1. DL - all vehicle types
2. RC - all vehicle types
3. Aadhaar - all vehicle types
4. Permit - Auto and Cab only
5. Fitness - all vehicle types
6. Insurance - all vehicle types
7. Approved - final approval outcome

Permit is treated as conditional because it is not required for ERickshaw captains.

The final Approved stage is taken directly from `final_status == "approved"` in the approval data rather than inferred solely from document indicators.


In [7]:
funnel["stage_dl"] = funnel["DL_cleared"]

In [8]:
funnel["stage_rc"] = (
    funnel["DL_cleared"]
    & funnel["RC_cleared"]
)

In [9]:
funnel["stage_aadhaar"] = (
    funnel["DL_cleared"]
    & funnel["RC_cleared"]
    & funnel["AADHAAR_cleared"]
)

In [10]:
permit_required = funnel["vehicle_type"].isin(["Auto", "Cab"])

In [11]:
funnel["stage_permit"] = (
    funnel["stage_aadhaar"]
    & (
        (~permit_required)
        | funnel["PERMIT_cleared"]
    )
)

In [12]:
funnel["stage_fitness"] = (
    funnel["stage_permit"]
    & funnel["FITNESS_cleared"]
)

In [13]:
funnel["stage_insurance"] = (
    funnel["stage_fitness"]
    & funnel["INSURANCE_cleared"]
)

In [14]:
funnel["stage_approved"] = (
    funnel["final_status"] == "approved"
)

In [15]:
stage_columns = [
    "stage_dl",
    "stage_rc",
    "stage_aadhaar",
    "stage_permit",
    "stage_fitness",
    "stage_insurance",
    "stage_approved"
]

In [17]:
stage_counts = funnel[stage_columns].sum()
stage_counts

stage_dl           20136
stage_rc           14647
stage_aadhaar      13093
stage_permit       10439
stage_fitness       7752
stage_insurance     4412
stage_approved      3998
dtype: int64

In [18]:
stage_counts.diff().dropna()

stage_rc          -5489.0
stage_aadhaar     -1554.0
stage_permit      -2654.0
stage_fitness     -2687.0
stage_insurance   -3340.0
stage_approved     -414.0
dtype: float64

In [19]:
print(
    "Permit required:",
    permit_required.sum()
)

Permit required: 18416


In [20]:
print(
    "Permit not required:",
    (~permit_required).sum()
)

Permit not required: 4341


## 4. Funnel Performance

For each stage, the analysis reports:

* **Stage volume:** number of captains reaching the stage.
* **Stage conversion:** percentage of captains from the immediately preceding stage who reach the current stage.
* **Cumulative conversion:** percentage of the original funnel cohort reaching the stage.
* **Volume lost:** number of captains lost between the previous stage and the current stage.

Stage-to-stage conversion is used to distinguish a large leak caused by a large upstream population from a stage with a particularly poor conversion rate.


In [21]:
funnel_table = pd.DataFrame({
    "stage": [
        "Signup",
        "DL",
        "RC",
        "Aadhaar",
        "Permit",
        "Fitness",
        "Insurance",
        "Approved"
    ],
    "captains": [
        len(funnel),
        funnel["stage_dl"].sum(),
        funnel["stage_rc"].sum(),
        funnel["stage_aadhaar"].sum(),
        funnel["stage_permit"].sum(),
        funnel["stage_fitness"].sum(),
        funnel["stage_insurance"].sum(),
        funnel["stage_approved"].sum()
    ]
})

In [25]:
funnel_table["volume_lost"] = (
    funnel_table["captains"].shift(1)
    - funnel_table["captains"]
)

funnel_table.loc[0, "volume_lost"] = np.nan

In [26]:
funnel_table["stage_conversion_pct"] = (
    funnel_table["captains"]
    / funnel_table["captains"].shift(1)
    * 100
)

funnel_table.loc[0, "stage_conversion_pct"] = 100

In [27]:
funnel_table["cumulative_conversion_pct"] = (
    funnel_table["captains"]
    / len(funnel)
    * 100
)

In [28]:
funnel_table

,stage,captains,volume_lost,stage_conversion_pct,cumulative_conversion_pct
0,Signup,22757,NaN,100.000000,100.000000
1,DL,20136,2621.0,88.482665,88.482665
2,RC,14647,5489.0,72.740366,64.362614
3,Aadhaar,13093,1554.0,89.390319,57.533946
4,Permit,10439,2654.0,79.729627,45.871600
5,Fitness,7752,2687.0,74.259987,34.064244
6,Insurance,4412,3340.0,56.914345,19.387441
7,Approved,3998,414.0,90.616500,17.568221


## 5. Stage Drop-off Analysis

The initial funnel shows substantial attrition throughout onboarding.

The largest absolute loss occurs between DL and RC, with **5,489 captains** not progressing to RC. However, the lowest stage-to-stage conversion occurs at Insurance, where only **56.9%** of captains reaching Fitness progress to Insurance.

These metrics alone do not identify the best intervention. The next analysis therefore examines the characteristics of captains lost at each stage and the underlying document verification outcomes to distinguish large but potentially unavoidable losses from actionable onboarding friction.


## 5.1 Verification Failure Reasons

The captain-level dataset captures whether documents were successfully cleared, but the underlying `doc_events` table contains the detailed reasons associated with verification failures.

The document-event data is therefore loaded separately for this diagnostic analysis. This allows failure reasons to be examined without changing the one-row-per-captain structure of the main funnel dataset.


In [31]:
doc_events = pd.read_csv(
    "../Data/doc_events.csv",
    parse_dates=["event_ts"]
)

In [32]:
failure_summary = (
    doc_events[
        doc_events["event_type"] == "verification_fail"
    ]["failure_reason"]
    .value_counts()
)

failure_summary

failure_reason
image_blurred          4767
ocr_low_confidence     3696
name_mismatch          3173
document_expired       2686
details_not_legible    2171
wrong_document_type    2083
duplicate_document     1196
Name: count, dtype: int64

In [33]:
failure_summary_pct = (
    failure_summary
    / failure_summary.sum()
    * 100
)

failure_summary_pct

failure_reason
image_blurred          24.109852
ocr_low_confidence     18.693101
name_mismatch          16.047947
document_expired       13.584867
details_not_legible    10.980174
wrong_document_type    10.535100
duplicate_document      6.048958
Name: count, dtype: float64

## 5.2 Verification Failures by Document

Overall failure reasons do not directly identify the largest onboarding leak because a failed verification can be corrected through a subsequent attempt.

To connect verification friction with the funnel, failure events are therefore examined by document type. This helps identify which onboarding stage experiences the greatest amount of verification friction before investigating whether that friction translates into captain drop-off.


In [34]:
failure_by_doc = (
    doc_events[
        doc_events["event_type"] == "verification_fail"
    ]
    .groupby("doc_type")
    .size()
    .sort_values(ascending=False)
)

failure_by_doc

doc_type
RC           8174
DL           3408
FITNESS      2693
INSURANCE    2126
PERMIT       1921
AADHAAR      1450
dtype: int64

In [35]:
failure_by_doc_pct = (
    failure_by_doc
    / failure_by_doc.sum()
    * 100
)

failure_by_doc_pct

doc_type
RC           41.341291
DL           17.236496
FITNESS      13.620271
INSURANCE    10.752579
PERMIT        9.715760
AADHAAR       7.333603
dtype: float64

In [36]:
failure_doc_reason = pd.crosstab(
    doc_events.loc[
        doc_events["event_type"] == "verification_fail",
        "doc_type"
    ],
    doc_events.loc[
        doc_events["event_type"] == "verification_fail",
        "failure_reason"
    ]
)

failure_doc_reason

failure_reason,details_not_legible,document_expired,duplicate_document,image_blurred,name_mismatch,ocr_low_confidence,wrong_document_type
doc_type,,,,,,,
AADHAAR,164,233,107,266,304,196,180
DL,428,531,239,624,693,497,396
FITNESS,281,301,150,750,386,574,251
INSURANCE,272,244,106,569,304,417,214
PERMIT,189,467,141,270,362,239,253
RC,837,910,453,2288,1124,1773,789


## 5.3 Captain-Level Stage Drop-off

Verification failures are event-level observations and may include multiple attempts by the same captain. To determine where captains actually leave the onboarding funnel, the analysis uses the cumulative stage indicators created earlier.

For each stage, the number of captains who reached the previous stage but did not reach the current stage is calculated. This separates repeated verification attempts from actual captain-level funnel attrition.


In [37]:
stage_pairs = [
    ("Signup", None, "stage_dl"),
    ("DL", "stage_dl", "stage_rc"),
    ("RC", "stage_rc", "stage_aadhaar"),
    ("Aadhaar", "stage_aadhaar", "stage_permit"),
    ("Permit", "stage_permit", "stage_fitness"),
    ("Fitness", "stage_fitness", "stage_insurance"),
    ("Insurance", "stage_insurance", "stage_approved")
]

In [38]:
dropoff_analysis = []

for stage, previous_stage, current_stage in stage_pairs:
    
    if previous_stage is None:
        previous_count = len(funnel)
        current_count = funnel[current_stage].sum()
    else:
        previous_count = funnel[previous_stage].sum()
        current_count = funnel[current_stage].sum()
    
    dropoff = previous_count - current_count
    conversion = current_count / previous_count * 100
    
    dropoff_analysis.append({
        "stage_transition": (
            f"{stage} → {current_stage.replace('stage_', '').title()}"
        ),
        "previous_stage_captains": previous_count,
        "current_stage_captains": current_count,
        "captains_lost": dropoff,
        "stage_conversion_pct": conversion
    })

dropoff_analysis = pd.DataFrame(dropoff_analysis)

dropoff_analysis

,stage_transition,previous_stage_captains,current_stage_captains,captains_lost,stage_conversion_pct
0,Signup → Dl,22757,20136,2621,88.482665
1,DL → Rc,20136,14647,5489,72.740366
2,RC → Aadhaar,14647,13093,1554,89.390319
3,Aadhaar → Permit,13093,10439,2654,79.729627
4,Permit → Fitness,10439,7752,2687,74.259987
5,Fitness → Insurance,7752,4412,3340,56.914345
6,Insurance → Approved,4412,3998,414,90.616500


In [39]:
rc_dropoff = funnel[
    funnel["stage_dl"]
    & ~funnel["stage_rc"]
].copy()

In [40]:
print("Captains reaching DL:", funnel["stage_dl"].sum())
print("Captains not progressing to RC:", len(rc_dropoff))

Captains reaching DL: 20136
Captains not progressing to RC: 5489


In [41]:
rc_dropoff["city"].value_counts()

city
Hyderabad    1707
Delhi        1401
Bangalore    1205
Pune         1176
Name: count, dtype: int64

In [42]:
rc_dropoff["vehicle_type"].value_counts()

vehicle_type
Auto         2477
Cab          1984
ERickshaw    1028
Name: count, dtype: int64

In [43]:
rc_dropoff["acquisition_channel"].value_counts()

acquisition_channel
organic_app       1880
referral          1278
fos_field         1129
paid_digital       880
gc_telecalling     322
Name: count, dtype: int64

In [44]:
rc_dropoff["device_tier"].value_counts()

device_tier
low     3044
mid     1810
high     635
Name: count, dtype: int64

## 5.4 Segment-Level RC Drop-off

The absolute number of RC drop-offs is influenced by the size of each segment. To distinguish high-volume segments from genuinely poor-performing segments, RC progression is evaluated using both volume and drop-off rate.

For each segment, the analysis compares captains who reached DL with those who subsequently progressed to RC.

The following dimensions are examined:

* City
* Vehicle type
* Acquisition channel
* Device tier

The RC drop-off rate is defined as:

**RC drop-off rate = (DL-stage captains − RC-stage captains) / DL-stage captains**


In [45]:
def stage_dropoff_by_segment(df, group_col, previous_stage, current_stage):
    grouped = (
        df.groupby(group_col)
        .agg(
            previous_stage_captains=(previous_stage, "sum"),
            current_stage_captains=(current_stage, "sum")
        )
        .reset_index()
    )

    grouped["captains_lost"] = (
        grouped["previous_stage_captains"]
        - grouped["current_stage_captains"]
    )

    grouped["dropoff_rate_pct"] = (
        grouped["captains_lost"]
        / grouped["previous_stage_captains"]
        * 100
    )

    grouped["stage_conversion_pct"] = (
        grouped["current_stage_captains"]
        / grouped["previous_stage_captains"]
        * 100
    )

    return grouped.sort_values(
        "dropoff_rate_pct",
        ascending=False
    )

In [46]:
rc_by_city = stage_dropoff_by_segment(
    funnel,
    "city",
    "stage_dl",
    "stage_rc"
)

rc_by_city

,city,previous_stage_captains,current_stage_captains,captains_lost,dropoff_rate_pct,stage_conversion_pct
2,Hyderabad,6151,4444,1707,27.751585,72.248415
0,Bangalore,4372,3167,1205,27.561757,72.438243
1,Delhi,5211,3810,1401,26.885435,73.114565
3,Pune,4402,3226,1176,26.715129,73.284871


In [47]:
rc_by_vehicle = stage_dropoff_by_segment(
    funnel,
    "vehicle_type",
    "stage_dl",
    "stage_rc"
)

rc_by_vehicle

,vehicle_type,previous_stage_captains,current_stage_captains,captains_lost,dropoff_rate_pct,stage_conversion_pct
1,Cab,7153,5169,1984,27.736614,72.263386
0,Auto,9144,6667,2477,27.088801,72.911199
2,ERickshaw,3839,2811,1028,26.777807,73.222193


In [48]:
rc_by_channel = stage_dropoff_by_segment(
    funnel,
    "acquisition_channel",
    "stage_dl",
    "stage_rc"
)

rc_by_channel

,acquisition_channel,previous_stage_captains,current_stage_captains,captains_lost,dropoff_rate_pct,stage_conversion_pct
3,paid_digital,2656,1776,880,33.132530,66.867470
4,referral,4437,3159,1278,28.803245,71.196755
1,gc_telecalling,1134,812,322,28.395062,71.604938
2,organic_app,6670,4790,1880,28.185907,71.814093
0,fos_field,5239,4110,1129,21.549914,78.450086


In [49]:
rc_by_device = stage_dropoff_by_segment(
    funnel,
    "device_tier",
    "stage_dl",
    "stage_rc"
)

rc_by_device

,device_tier,previous_stage_captains,current_stage_captains,captains_lost,dropoff_rate_pct,stage_conversion_pct
1,low,9279,6235,3044,32.805259,67.194741
2,mid,7739,5929,1810,23.388035,76.611965
0,high,3118,2483,635,20.365619,79.634381


## 5.5 Compare Major Funnel Leaks

The RC stage has the largest absolute captain loss, while Insurance has the lowest stage-to-stage conversion.

To determine which represents the more actionable opportunity, the same segment-level analysis is applied to the Insurance transition. This allows the analysis to compare not only the size of each leak, but also whether poor performance is concentrated in identifiable and potentially addressable segments.


In [50]:
insurance_dropoff = funnel[
    funnel["stage_fitness"]
    & ~funnel["stage_insurance"]
].copy()

In [51]:
print(
    "Captains reaching Fitness:",
    funnel["stage_fitness"].sum()
)

Captains reaching Fitness: 7752


In [52]:
print(
    "Captains not progressing to Insurance:",
    len(insurance_dropoff)
)

Captains not progressing to Insurance: 3340


In [53]:
insurance_by_city = stage_dropoff_by_segment(
    funnel,
    "city",
    "stage_fitness",
    "stage_insurance"
)

insurance_by_city

,city,previous_stage_captains,current_stage_captains,captains_lost,dropoff_rate_pct,stage_conversion_pct
3,Pune,1584,890,694,43.813131,56.186869
1,Delhi,2219,1254,965,43.488058,56.511942
2,Hyderabad,2302,1319,983,42.701998,57.298002
0,Bangalore,1647,949,698,42.380085,57.619915


In [54]:
insurance_by_vehicle = stage_dropoff_by_segment(
    funnel,
    "vehicle_type",
    "stage_fitness",
    "stage_insurance"
)

insurance_by_vehicle

,vehicle_type,previous_stage_captains,current_stage_captains,captains_lost,dropoff_rate_pct,stage_conversion_pct
1,Cab,2588,1444,1144,44.204019,55.795981
0,Auto,3271,1844,1427,43.625803,56.374197
2,ERickshaw,1893,1124,769,40.623349,59.376651


In [55]:
insurance_by_channel = stage_dropoff_by_segment(
    funnel,
    "acquisition_channel",
    "stage_fitness",
    "stage_insurance"
)

insurance_by_channel

,acquisition_channel,previous_stage_captains,current_stage_captains,captains_lost,dropoff_rate_pct,stage_conversion_pct
3,paid_digital,740,345,395,53.378378,46.621622
2,organic_app,2363,1281,1082,45.789251,54.210749
4,referral,1613,892,721,44.699318,55.300682
1,gc_telecalling,411,237,174,42.335766,57.664234
0,fos_field,2625,1657,968,36.876190,63.123810


In [56]:
insurance_by_device = stage_dropoff_by_segment(
    funnel,
    "device_tier",
    "stage_fitness",
    "stage_insurance"
)

insurance_by_device

,device_tier,previous_stage_captains,current_stage_captains,captains_lost,dropoff_rate_pct,stage_conversion_pct
1,low,3097,1613,1484,47.917339,52.082661
2,mid,3225,1896,1329,41.209302,58.790698
0,high,1430,903,527,36.853147,63.146853


## 5.6 Attempt and Recovery Analysis

A verification failure does not necessarily represent permanent funnel loss because captains can re-upload a document up to three times.

To distinguish recoverable verification friction from permanent attrition, document attempts are examined to determine how often a failed document is subsequently cleared.

This helps assess whether the observed funnel leak is primarily caused by verification friction or by captains abandoning onboarding despite having the opportunity to retry.


In [57]:
doc_attempts = (
    doc_events
    .groupby(["captain_id", "doc_type"])
    .agg(
        max_attempt=("attempt_no", "max"),
        passed=("event_type", lambda x: (x == "verification_pass").any()),
        failed=("event_type", lambda x: (x == "verification_fail").any())
    )
    .reset_index()
)

In [58]:
failed_docs = doc_attempts[
    doc_attempts["failed"]
].copy()

In [59]:
recovery_summary = (
    failed_docs
    .groupby("doc_type")
    .agg(
        failed_document_instances=("failed", "sum"),
        eventually_cleared=("passed", "sum")
    )
)

In [60]:
recovery_summary["recovery_rate_pct"] = (
    recovery_summary["eventually_cleared"]
    / recovery_summary["failed_document_instances"]
    * 100
)

In [61]:
recovery_summary.sort_values(
    "recovery_rate_pct",
    ascending=False
)

,failed_document_instances,eventually_cleared,recovery_rate_pct
doc_type,,,
AADHAAR,1439,894,62.126477
DL,3308,2011,60.792019
PERMIT,1744,1027,58.887615
FITNESS,2362,1369,57.959356
INSURANCE,1785,966,54.117647
RC,6702,3344,49.895554


## 5.7 Estimate RC Recovery Opportunity

The RC stage combines the largest captain-level drop-off with the highest volume of failed document instances and the lowest recovery rate among the major document stages.

To quantify the potential opportunity, a scenario is constructed for improving RC progression. This is an illustrative sizing exercise rather than a causal estimate: it assumes that a specified fraction of current RC drop-offs could be recovered while all downstream conversion rates remain unchanged.


In [62]:
rc_to_approval_rate = (
    funnel["stage_approved"].sum()
    / funnel["stage_rc"].sum()
)

rc_to_approval_rate

0.27295691950570083

In [63]:
print(
    "RC → Approved conversion:",
    round(rc_to_approval_rate * 100, 2),
    "%"
)

RC → Approved conversion: 27.3 %


In [64]:
rc_dropoff_count = (
    funnel["stage_dl"].sum()
    - funnel["stage_rc"].sum()
)

additional_rc_captains = rc_dropoff_count * 0.10

additional_approved = (
    additional_rc_captains
    * rc_to_approval_rate
)

print("Current RC drop-off:", rc_dropoff_count)
print("10% RC recovery:", additional_rc_captains)
print(
    "Estimated additional approvals:",
    round(additional_approved, 0)
)

Current RC drop-off: 5489
10% RC recovery: 548.9
Estimated additional approvals: 150.0


The scenario implies that recovering 10% of current RC drop-offs could generate approximately **150 additional approvals**, assuming recovered captains convert from RC to approval at the current observed rate. This would increase A2O from approximately **17.57% to 18.23%**, or about **+0.66 percentage points**.

This should be treated as an illustrative opportunity-sizing scenario rather than a causal forecast, since the analysis does not establish that a specific intervention can recover 10% of RC drop-offs.


## 5.8 Diagnose RC Failure Modes

RC is the largest captain-level funnel drop-off. The next step is to examine verification failure reasons specifically for RC to identify whether the leakage is concentrated in particular document-quality or verification problems.

This analysis uses verification failure events rather than captain-level drop-off counts. A single captain may generate multiple failure events, so these figures measure verification friction rather than unique captains lost.


In [65]:
rc_failures = doc_events[
    (doc_events["doc_type"] == "RC") &
    (doc_events["event_type"] == "verification_fail")
].copy()

In [66]:
rc_failure_reasons = (
    rc_failures["failure_reason"]
    .value_counts()
    .rename_axis("failure_reason")
    .reset_index(name="failure_events")
)

In [67]:
rc_failure_reasons["failure_share_pct"] = (
    rc_failure_reasons["failure_events"]
    / rc_failure_reasons["failure_events"].sum()
    * 100
)

In [68]:
rc_failure_reasons

,failure_reason,failure_events,failure_share_pct
0,image_blurred,2288,27.991192
1,ocr_low_confidence,1773,21.690727
2,name_mismatch,1124,13.750918
3,document_expired,910,11.132860
4,details_not_legible,837,10.239785
5,wrong_document_type,789,9.652557
6,duplicate_document,453,5.541962


In [69]:
print("Total RC failure events:", len(rc_failures))
print("\nRC failure reasons:")
for _, row in rc_failure_reasons.iterrows():
    print(
        f'{row["failure_reason"]}: '
        f'{int(row["failure_events"]):,} '
        f'({row["failure_share_pct"]:.2f}%)'
    )

Total RC failure events: 8174

RC failure reasons:
image_blurred: 2,288 (27.99%)
ocr_low_confidence: 1,773 (21.69%)
name_mismatch: 1,124 (13.75%)
document_expired: 910 (11.13%)
details_not_legible: 837 (10.24%)
wrong_document_type: 789 (9.65%)
duplicate_document: 453 (5.54%)


## 5.9 RC Failure Recovery by Reason

Failure volume alone does not establish that a problem is actionable. To assess potential recovery, RC failures are linked to the eventual outcome of each captain-RC document instance.

The analysis measures the share of failed RC document instances that eventually achieved a verification pass. This helps distinguish high-volume failure modes from failure modes where recovery appears feasible.

The result is interpreted as an observed recovery pattern, not a causal estimate of intervention impact.


In [70]:
rc_attempts = (
    doc_events[doc_events["doc_type"] == "RC"]
    .groupby(["captain_id", "doc_type"])
    .agg(
        max_attempt=("attempt_no", "max"),
        passed=("event_type", lambda x: (x == "verification_pass").any()),
        failed=("event_type", lambda x: (x == "verification_fail").any())
    )
    .reset_index()
)

In [71]:
rc_failed_instances = rc_attempts[
    rc_attempts["failed"]
].copy()

In [72]:
rc_failed_instances.shape

(6702, 5)

In [73]:
rc_first_failure = (
    doc_events[
        (doc_events["doc_type"] == "RC") &
        (doc_events["event_type"] == "verification_fail")
    ]
    .sort_values(["captain_id", "event_ts"])
    .drop_duplicates(["captain_id"], keep="first")
    [["captain_id", "failure_reason"]]
)

In [74]:
rc_recovery_by_reason = (
    rc_failed_instances
    .merge(rc_first_failure, on="captain_id", how="left")
    .groupby("failure_reason")
    .agg(
        failed_instances=("failed", "sum"),
        eventually_cleared=("passed", "sum")
    )
    .reset_index()
)

In [75]:
rc_recovery_by_reason["recovery_rate_pct"] = (
    rc_recovery_by_reason["eventually_cleared"]
    / rc_recovery_by_reason["failed_instances"]
    * 100
)

In [76]:
rc_recovery_by_reason = rc_recovery_by_reason.sort_values(
    "recovery_rate_pct",
    ascending=False
)

In [77]:
rc_recovery_by_reason

,failure_reason,failed_instances,eventually_cleared,recovery_rate_pct
4,name_mismatch,960,510,53.125000
1,document_expired,759,398,52.437418
0,details_not_legible,676,348,51.479290
6,wrong_document_type,666,328,49.249249
5,ocr_low_confidence,1421,689,48.486981
3,image_blurred,1842,891,48.371336
2,duplicate_document,378,180,47.619048


## 5.10 RC Image/OCR Opportunity

Image quality and OCR confidence are the two largest observed RC failure categories. Together they account for a substantial share of failed RC document instances.

These failure modes are potentially addressable through product interventions such as clearer capture guidance, image-quality checks before submission, or prompting the captain to retake an image when the document is unlikely to be readable.

The analysis does not assume that all such failures are recoverable. Instead, the affected population is treated as an opportunity pool for intervention testing.


In [78]:
image_ocr_instances = rc_recovery_by_reason[
    rc_recovery_by_reason["failure_reason"].isin(
        ["image_blurred", "ocr_low_confidence"]
    )
].copy()

image_ocr_instances

,failure_reason,failed_instances,eventually_cleared,recovery_rate_pct
5,ocr_low_confidence,1421,689,48.486981
3,image_blurred,1842,891,48.371336


In [79]:
image_ocr_failed = image_ocr_instances["failed_instances"].sum()
image_ocr_recovered = image_ocr_instances["eventually_cleared"].sum()

image_ocr_recovery_rate = (
    image_ocr_recovered / image_ocr_failed * 100
)

print("Image/OCR failed RC instances:", image_ocr_failed)
print("Eventually cleared:", image_ocr_recovered)
print("Observed recovery rate:", round(image_ocr_recovery_rate, 2), "%")

Image/OCR failed RC instances: 3263
Eventually cleared: 1580
Observed recovery rate: 48.42 %


## 6. R2A: Approval to First Order

The second funnel examines the transition from final approval to the captain's first completed order.

Unlike the A2O funnel, this analysis starts from captains who reached final approval. The objective is to measure post-approval activation, identify meaningful differences across segments, and determine whether there is an actionable opportunity after onboarding.


In [80]:
approved = captain_base[
    captain_base["final_status"] == "approved"
].copy()

approved["first_order_completed"] = (
    approved["first_order_ts"].notna()
)

print("Approved captains:", len(approved))
print(
    "Approved with first order:",
    approved["first_order_completed"].sum()
)
print(
    "Approved without first order:",
    (~approved["first_order_completed"]).sum()
)

r2a_rate = (
    approved["first_order_completed"].mean() * 100
)

print("R2A conversion:", round(r2a_rate, 2), "%")

Approved captains: 4206
Approved with first order: 4104
Approved without first order: 102
R2A conversion: 97.57 %


In [81]:
time_to_first_order = approved[
    approved["first_order_ts"].notna()
]["approval_to_first_order_days"]

print(time_to_first_order.describe())

count    4104.000000
mean        1.714744
std         1.365636
min         0.001303
25%         0.721890
50%         1.374499
75%         2.364522
max        11.068693
Name: approval_to_first_order_days, dtype: float64


## 6.2 R2A by Acquisition Channel

Overall R2A conversion is very high, so the next check is whether post-approval activation varies materially across acquisition channels.

This is a diagnostic segmentation rather than a causal comparison. The objective is to identify whether any acquisition channel has a sufficiently large activation gap to warrant further investigation.


In [82]:
r2a_by_channel = (
    approved
    .groupby("acquisition_channel")
    .agg(
        approved_captains=("captain_id", "count"),
        first_orders=("first_order_completed", "sum")
    )
    .reset_index()
)

r2a_by_channel["r2a_rate_pct"] = (
    r2a_by_channel["first_orders"]
    / r2a_by_channel["approved_captains"]
    * 100
)

r2a_by_channel = r2a_by_channel.sort_values(
    "r2a_rate_pct"
)

r2a_by_channel

,acquisition_channel,approved_captains,first_orders,r2a_rate_pct
1,gc_telecalling,206,199,96.601942
3,paid_digital,341,330,96.774194
0,fos_field,1586,1539,97.036570
2,organic_app,1220,1198,98.196721
4,referral,853,838,98.241501


In [83]:
r2a_by_city = (
    approved
    .groupby("city")
    .agg(
        approved_captains=("captain_id", "count"),
        first_orders=("first_order_completed", "sum")
    )
    .reset_index()
)

r2a_by_city["r2a_rate_pct"] = (
    r2a_by_city["first_orders"]
    / r2a_by_city["approved_captains"]
    * 100
)

r2a_by_city.sort_values("r2a_rate_pct")

,city,approved_captains,first_orders,r2a_rate_pct
2,Hyderabad,1264,1225,96.914557
1,Delhi,1206,1172,97.180763
0,Bangalore,897,879,97.993311
3,Pune,839,828,98.688915


## 7. Supply Analysis

The final analytical area examines supply performance using the airport hourly and trip-level datasets.

The objective is to identify whether there are meaningful spatial or temporal supply gaps and, if so, whether the data supports an actionable intervention.

Because the airport datasets cover **May–June 2026**, while captain onboarding data covers the full January–June period, supply analysis is treated as a separate marketplace view rather than directly combining it with the onboarding funnel.


In [87]:
airport_hourly = pd.read_csv(
    "../Data/airport_hourly.csv",
    parse_dates=["hour_ts"]
)

airport_trips = pd.read_csv("../Data/airport_trips.csv")

print("Airport trips shape:", airport_trips.shape)

print("\nAirport trips columns:")
print(airport_trips.columns.tolist())

print("\nAirport trips:")
display(airport_trips.head())

print("Airport hourly shape:", airport_hourly.shape)
print("Airport trips shape:", airport_trips.shape)

print("\nAirport hourly columns:")
print(airport_hourly.columns.tolist())

print("\nAirport trips columns:")
print(airport_trips.columns.tolist())

Airport trips shape: (60000, 9)

Airport trips columns:
['trip_id', 'pickup_zone_id', 'drop_zone_id', 'drop_zone_type', 'request_ts', 'trip_distance_km', 'captain_cancelled', 'got_return_fare_within_20min', 'fare_inr']

Airport trips:


,trip_id,pickup_zone_id,drop_zone_id,drop_zone_type,request_ts,trip_distance_km,captain_cancelled,got_return_fare_within_20min,fare_inr
0,TRP00036169,APT-T1,CBD-02,city_core,2026-05-01 00:00:00,8.18,0,1,148.0
1,TRP00018764,APT-T1,SUB-07,suburban,2026-05-01 00:00:00,21.10,0,0,302.0
2,TRP00027350,APT-T2,TECH-03,tech_park,2026-05-01 00:00:00,14.22,0,0,231.0
3,TRP00058214,APT-T2,TECH-03,tech_park,2026-05-01 00:00:00,9.81,0,0,207.0
4,TRP00044709,APT-T1,SUB-07,suburban,2026-05-01 00:00:00,18.02,1,0,333.0


Airport hourly shape: (10248, 9)
Airport trips shape: (60000, 9)

Airport hourly columns:
['zone_id', 'zone_type', 'hour_ts', 'requests', 'fulfilled_requests', 'unfulfilled_requests', 'online_captains', 'avg_eta_min', 'avg_surge_multiplier']

Airport trips columns:
['trip_id', 'pickup_zone_id', 'drop_zone_id', 'drop_zone_type', 'request_ts', 'trip_distance_km', 'captain_cancelled', 'got_return_fare_within_20min', 'fare_inr']


In [88]:
print("\nAirport hourly:")
display(airport_hourly.head())

print("\nAirport trips:")
display(airport_trips.head())


Airport hourly:


,zone_id,zone_type,hour_ts,requests,fulfilled_requests,unfulfilled_requests,online_captains,avg_eta_min,avg_surge_multiplier
0,APT-T1,airport_terminal,2026-05-01 00:00:00,83,31,52,15,9.13,2.04
1,APT-T1,airport_terminal,2026-05-01 01:00:00,111,27,84,10,10.13,2.17
2,APT-T1,airport_terminal,2026-05-01 02:00:00,86,21,65,13,10.89,2.19
3,APT-T1,airport_terminal,2026-05-01 03:00:00,54,25,29,15,9.27,1.80
4,APT-T1,airport_terminal,2026-05-01 04:00:00,36,34,2,17,4.54,1.08



Airport trips:


,trip_id,pickup_zone_id,drop_zone_id,drop_zone_type,request_ts,trip_distance_km,captain_cancelled,got_return_fare_within_20min,fare_inr
0,TRP00036169,APT-T1,CBD-02,city_core,2026-05-01 00:00:00,8.18,0,1,148.0
1,TRP00018764,APT-T1,SUB-07,suburban,2026-05-01 00:00:00,21.10,0,0,302.0
2,TRP00027350,APT-T2,TECH-03,tech_park,2026-05-01 00:00:00,14.22,0,0,231.0
3,TRP00058214,APT-T2,TECH-03,tech_park,2026-05-01 00:00:00,9.81,0,0,207.0
4,TRP00044709,APT-T1,SUB-07,suburban,2026-05-01 00:00:00,18.02,1,0,333.0


In [90]:
print("\nAirport trip timestamp range:")
print(
    airport_trips["request_ts"].min(),
    "to",
    airport_trips["request_ts"].max()
)

print("\nHourly zones:")
print(airport_hourly["zone_id"].nunique())

print("\nAirport trip count:")
print(airport_trips["trip_id"].nunique())


Airport trip timestamp range:
2026-05-01 00:00:00 to 2026-06-30 23:00:00

Hourly zones:
7

Airport trip count:
60000


## 7.2 Overall Airport Marketplace Performance

The hourly airport dataset provides the primary view of marketplace balance. The key measures are request fulfillment, unfulfilled demand, captain availability, ETA, and surge.

The first step is to establish overall marketplace performance before examining differences across zones or time periods.


In [91]:
total_requests = airport_hourly["requests"].sum()
total_fulfilled = airport_hourly["fulfilled_requests"].sum()
total_unfulfilled = airport_hourly["unfulfilled_requests"].sum()

fulfillment_rate = (
    total_fulfilled / total_requests * 100
)

unfulfilled_rate = (
    total_unfulfilled / total_requests * 100
)

print("Total requests:", total_requests)
print("Fulfilled requests:", total_fulfilled)
print("Unfulfilled requests:", total_unfulfilled)
print("Fulfillment rate:", round(fulfillment_rate, 2), "%")
print("Unfulfilled rate:", round(unfulfilled_rate, 2), "%")

Total requests: 457610
Fulfilled requests: 393269
Unfulfilled requests: 64341
Fulfillment rate: 85.94 %
Unfulfilled rate: 14.06 %


In [92]:
print("Average online captains:", round(
    airport_hourly["online_captains"].mean(), 2
))

print("Median online captains:", round(
    airport_hourly["online_captains"].median(), 2
))

print("Average ETA:", round(
    airport_hourly["avg_eta_min"].mean(), 2
), "min")

print("Average surge multiplier:", round(
    airport_hourly["avg_surge_multiplier"].mean(), 2
))

Average online captains: 36.25
Median online captains: 34.0
Average ETA: 4.27 min
Average surge multiplier: 1.15


In [93]:
airport_hourly["unfulfilled_rate_pct"] = (
    airport_hourly["unfulfilled_requests"]
    / airport_hourly["requests"]
    * 100
)

airport_hourly[
    "unfulfilled_rate_pct"
].describe()

count    10248.000000
mean         9.926309
std         18.732266
min          1.515152
25%          2.325581
50%          2.941176
75%          4.545455
max         76.363636
Name: unfulfilled_rate_pct, dtype: float64

## 7.3 Supply Pressure by Zone

Overall fulfillment masks potential differences across locations. The next step compares zones using request volume, fulfillment, unfulfilled demand, captain availability, ETA, and surge.

The objective is to identify whether supply pressure is concentrated in specific zones rather than uniformly distributed across the airport network.


In [94]:
supply_by_zone = (
    airport_hourly
    .groupby(["zone_id", "zone_type"])
    .agg(
        requests=("requests", "sum"),
        fulfilled_requests=("fulfilled_requests", "sum"),
        unfulfilled_requests=("unfulfilled_requests", "sum"),
        avg_online_captains=("online_captains", "mean"),
        avg_eta_min=("avg_eta_min", "mean"),
        avg_surge=("avg_surge_multiplier", "mean")
    )
    .reset_index()
)

supply_by_zone["fulfillment_rate_pct"] = (
    supply_by_zone["fulfilled_requests"]
    / supply_by_zone["requests"]
    * 100
)

supply_by_zone["unfulfilled_rate_pct"] = (
    supply_by_zone["unfulfilled_requests"]
    / supply_by_zone["requests"]
    * 100
)

supply_by_zone.sort_values(
    "unfulfilled_rate_pct",
    ascending=False
)

,zone_id,zone_type,requests,fulfilled_requests,unfulfilled_requests,avg_online_captains,avg_eta_min,avg_surge,fulfillment_rate_pct,unfulfilled_rate_pct
1,APT-T2,airport_terminal,68555,40865,27690,29.670765,5.763490,1.408299,59.609073,40.390927
0,APT-T1,airport_terminal,68259,40891,27368,29.901639,5.735594,1.403730,59.905653,40.094347
4,SUB-07,suburban,51833,50070,1763,31.282104,3.722705,1.043661,96.598692,3.401308
5,SUB-11,suburban,51444,49739,1705,31.284153,3.643245,1.043545,96.685717,3.314283
6,TECH-03,tech_park,61388,59531,1857,41.204918,3.654529,1.036926,96.974979,3.025021
3,CBD-02,city_core,78052,76067,1985,45.265710,3.689146,1.039740,97.456824,2.543176
2,CBD-01,city_core,78079,76106,1973,45.153005,3.667145,1.041728,97.473072,2.526928


In [95]:
supply_by_zone[
    [
        "zone_id",
        "zone_type",
        "requests",
        "unfulfilled_requests",
        "unfulfilled_rate_pct",
        "avg_online_captains",
        "avg_eta_min",
        "avg_surge"
    ]
].sort_values(
    "unfulfilled_requests",
    ascending=False
)

,zone_id,zone_type,requests,unfulfilled_requests,unfulfilled_rate_pct,avg_online_captains,avg_eta_min,avg_surge
1,APT-T2,airport_terminal,68555,27690,40.390927,29.670765,5.763490,1.408299
0,APT-T1,airport_terminal,68259,27368,40.094347,29.901639,5.735594,1.403730
3,CBD-02,city_core,78052,1985,2.543176,45.265710,3.689146,1.039740
2,CBD-01,city_core,78079,1973,2.526928,45.153005,3.667145,1.041728
6,TECH-03,tech_park,61388,1857,3.025021,41.204918,3.654529,1.036926
4,SUB-07,suburban,51833,1763,3.401308,31.282104,3.722705,1.043661
5,SUB-11,suburban,51444,1705,3.314283,31.284153,3.643245,1.043545


## 7.4 Airport Trip Behavior

The hourly data shows severe fulfillment pressure at the two airport-terminal zones. The trip-level data is used to investigate whether this pressure may be related to captain behavior after receiving airport-originating trips.

Two behaviors are examined: captain cancellation and whether a captain receives a return fare within 20 minutes. The latter provides a proxy for the attractiveness of remaining in the airport catchment after completing a trip.

These measures are descriptive and do not establish that cancellation or return-fare availability causes unfulfilled demand.


In [96]:
total_trips = len(airport_trips)

cancelled_trips = airport_trips["captain_cancelled"].sum()

cancellation_rate = (
    cancelled_trips / total_trips * 100
)

return_fare_trips = (
    airport_trips["got_return_fare_within_20min"].sum()
)

return_fare_rate = (
    return_fare_trips / total_trips * 100
)

print("Total airport trips:", total_trips)
print("Captain-cancelled trips:", cancelled_trips)
print("Captain cancellation rate:", round(cancellation_rate, 2), "%")
print("Trips with return fare within 20 min:", return_fare_trips)
print("Return-fare rate:", round(return_fare_rate, 2), "%")

Total airport trips: 60000
Captain-cancelled trips: 8260
Captain cancellation rate: 13.77 %
Trips with return fare within 20 min: 21576
Return-fare rate: 35.96 %


In [97]:
trip_by_destination = (
    airport_trips
    .groupby("drop_zone_type")
    .agg(
        trips=("trip_id", "count"),
        cancelled_trips=("captain_cancelled", "sum"),
        return_fare_trips=("got_return_fare_within_20min", "sum"),
        avg_distance_km=("trip_distance_km", "mean"),
        avg_fare_inr=("fare_inr", "mean")
    )
    .reset_index()
)

trip_by_destination["cancellation_rate_pct"] = (
    trip_by_destination["cancelled_trips"]
    / trip_by_destination["trips"]
    * 100
)

trip_by_destination["return_fare_rate_pct"] = (
    trip_by_destination["return_fare_trips"]
    / trip_by_destination["trips"]
    * 100
)

trip_by_destination

,drop_zone_type,trips,cancelled_trips,return_fare_trips,avg_distance_km,avg_fare_inr,cancellation_rate_pct,return_fare_rate_pct
0,city_core,25282,2181,13329,13.237335,216.623092,8.626691,52.721304
1,suburban,24643,5176,4073,23.376524,353.565475,21.003936,16.528020
2,tech_park,10075,903,4174,16.104781,255.227792,8.962779,41.429280


In [98]:
trip_by_pickup = (
    airport_trips
    .groupby("pickup_zone_id")
    .agg(
        trips=("trip_id", "count"),
        cancelled_trips=("captain_cancelled", "sum"),
        return_fare_trips=("got_return_fare_within_20min", "sum"),
        avg_distance_km=("trip_distance_km", "mean"),
        avg_fare_inr=("fare_inr", "mean")
    )
    .reset_index()
)

trip_by_pickup["cancellation_rate_pct"] = (
    trip_by_pickup["cancelled_trips"]
    / trip_by_pickup["trips"]
    * 100
)

trip_by_pickup["return_fare_rate_pct"] = (
    trip_by_pickup["return_fare_trips"]
    / trip_by_pickup["trips"]
    * 100
)

trip_by_pickup

,pickup_zone_id,trips,cancelled_trips,return_fare_trips,avg_distance_km,avg_fare_inr,cancellation_rate_pct,return_fare_rate_pct
0,APT-T1,30084,4126,10839,17.858662,279.068010,13.714932,36.029118
1,APT-T2,29916,4134,10737,17.907797,279.633541,13.818692,35.890493


## 7.5 Airport Trip Behavior: Interpretation

All **60,000 trips** in the trip-level dataset originate from the two airport-terminal zones (`APT-T1` and `APT-T2`). Therefore, this dataset does not provide a non-airport comparison group.

The two airport terminals show nearly identical trip behavior: captain cancellation is approximately **13.7%** and return-fare availability within 20 minutes is approximately **36%** at both terminals.

Because no non-airport control group is available, these metrics should be interpreted as descriptive characteristics of airport-originating trips rather than evidence that airport trips have unusually high cancellation or low return-fare availability.


In [101]:
print(
    "Unique pickup zones:",
    airport_trips["pickup_zone_id"].unique()
)

print(
    "All trips are airport-originating:",
    airport_trips["pickup_zone_id"].str.startswith("APT-").all()
)

print(
    "Number of pickup zones:",
    airport_trips["pickup_zone_id"].nunique()
)

Unique pickup zones: ['APT-T1' 'APT-T2']
All trips are airport-originating: True
Number of pickup zones: 2


## 7.6 Supply Pressure by Hour

The airport-terminal zones show substantially higher unfulfilled demand than the other observed zones. The next step is to determine whether this pressure is concentrated at particular hours of the day.

Hourly aggregation is used to identify recurring periods of high unfulfilled demand and to distinguish persistent supply pressure from isolated observations.


In [103]:
airport_hourly["hour"] = airport_hourly["hour_ts"].dt.hour

supply_by_hour = (
    airport_hourly
    .groupby("hour")
    .agg(
        requests=("requests", "sum"),
        fulfilled_requests=("fulfilled_requests", "sum"),
        unfulfilled_requests=("unfulfilled_requests", "sum"),
        avg_online_captains=("online_captains", "mean"),
        avg_eta_min=("avg_eta_min", "mean"),
        avg_surge=("avg_surge_multiplier", "mean")
    )
    .reset_index()
)

supply_by_hour["unfulfilled_rate_pct"] = (
    supply_by_hour["unfulfilled_requests"]
    / supply_by_hour["requests"]
    * 100
)

supply_by_hour["fulfillment_rate_pct"] = (
    supply_by_hour["fulfilled_requests"]
    / supply_by_hour["requests"]
    * 100
)

supply_by_hour.sort_values(
    "unfulfilled_rate_pct",
    ascending=False
)

,hour,requests,fulfilled_requests,unfulfilled_requests,avg_online_captains,avg_eta_min,avg_surge,unfulfilled_rate_pct,fulfillment_rate_pct
1,1,21560,13040,8520,25.360656,5.435105,1.360796,39.517625,60.482375
23,23,25739,15748,9991,30.461358,5.539485,1.362740,38.816582,61.183418
2,2,19703,12929,6774,25.714286,5.360890,1.331241,34.380551,65.619449
0,0,19378,12844,6534,25.543326,5.317237,1.336862,33.718650,66.281350
22,22,26086,17601,8485,34.526932,5.498009,1.358501,32.527026,67.472974
3,3,16363,12739,3624,25.641686,4.840820,1.256721,22.147528,77.852472
21,21,24012,19225,4787,38.681499,5.117377,1.296792,19.935865,80.064135
4,4,15314,13426,1888,26.810304,4.415597,1.167400,12.328588,87.671412
6,6,19203,16926,2277,32.236534,4.293208,1.152881,11.857522,88.142478
5,5,16466,14676,1790,28.845433,4.280375,1.143279,10.870885,89.129115


In [104]:
supply_by_hour[
    [
        "hour",
        "requests",
        "unfulfilled_requests",
        "unfulfilled_rate_pct",
        "avg_online_captains",
        "avg_eta_min",
        "avg_surge"
    ]
].sort_values(
    "unfulfilled_requests",
    ascending=False
)

,hour,requests,unfulfilled_requests,unfulfilled_rate_pct,avg_online_captains,avg_eta_min,avg_surge
23,23,25739,9991,38.816582,30.461358,5.539485,1.362740
1,1,21560,8520,39.517625,25.360656,5.435105,1.360796
22,22,26086,8485,32.527026,34.526932,5.498009,1.358501
2,2,19703,6774,34.380551,25.714286,5.360890,1.331241
0,0,19378,6534,33.718650,25.543326,5.317237,1.336862
21,21,24012,4787,19.935865,38.681499,5.117377,1.296792
3,3,16363,3624,22.147528,25.641686,4.840820,1.256721
6,6,19203,2277,11.857522,32.236534,4.293208,1.152881
4,4,15314,1888,12.328588,26.810304,4.415597,1.167400
5,5,16466,1790,10.870885,28.845433,4.280375,1.143279


In [105]:
airport_hourly["hour"] = airport_hourly["hour_ts"].dt.hour

airport_terminal = airport_hourly[
    airport_hourly["zone_id"].isin(["APT-T1", "APT-T2"])
].copy()

terminal_by_hour = (
    airport_terminal
    .groupby("hour")
    .agg(
        requests=("requests", "sum"),
        unfulfilled_requests=("unfulfilled_requests", "sum"),
        avg_online_captains=("online_captains", "mean"),
        avg_eta_min=("avg_eta_min", "mean"),
        avg_surge=("avg_surge_multiplier", "mean")
    )
    .reset_index()
)

terminal_by_hour["unfulfilled_rate_pct"] = (
    terminal_by_hour["unfulfilled_requests"]
    / terminal_by_hour["requests"]
    * 100
)

terminal_by_hour.sort_values(
    "unfulfilled_rate_pct",
    ascending=False
)

,hour,requests,unfulfilled_requests,avg_online_captains,avg_eta_min,avg_surge,unfulfilled_rate_pct
23,23,13007,9594,12.565574,10.041393,2.165246,73.760283
1,1,11288,8145,12.844262,9.826230,2.146066,72.156272
22,22,11188,8042,13.024590,9.900164,2.142951,71.880586
0,0,9124,6191,12.598361,9.540328,2.073033,67.854011
2,2,9500,6427,13.213115,9.577541,2.061557,67.652632
21,21,7293,4347,13.344262,8.769426,1.926230,59.605101
3,3,6190,3274,13.221311,8.005902,1.800574,52.891761
4,4,4718,1499,14.983607,6.175410,1.470574,31.771937
6,6,6776,1944,21.844262,5.881066,1.433770,28.689492
5,5,5291,1452,17.581967,5.831967,1.408443,27.442827


## 8. Campaign Evaluation — CAMP_WA_002

In [106]:
nudges = pd.read_csv("../Data/nudges.csv", parse_dates=["sent_ts"])

print("Shape:", nudges.shape)
print("\nColumns:")
print(nudges.columns.tolist())

print("\nCampaigns:")
print(nudges["campaign_id"].value_counts())

print("\nDelivery:")
print(nudges["delivered"].value_counts(dropna=False))

print("\nClicks:")
print(nudges["clicked"].value_counts(dropna=False))

Shape: (16314, 6)

Columns:
['captain_id', 'campaign_id', 'channel', 'sent_ts', 'delivered', 'clicked']

Campaigns:
campaign_id
CAMP_WA_002      8673
CAMP_WA_001      3807
CAMP_SMS_004     2661
CAMP_CALL_009    1173
Name: count, dtype: int64

Delivery:
delivered
1    14947
0     1367
Name: count, dtype: int64

Clicks:
clicked
0    10833
1     5481
Name: count, dtype: int64


In [107]:
camp_wa_002 = nudges[nudges["campaign_id"] == "CAMP_WA_002"].copy()

print("CAMP_WA_002 shape:", camp_wa_002.shape)

print("\nChannels:")
print(camp_wa_002["channel"].value_counts())

print("\nDelivery:")
print(camp_wa_002["delivered"].value_counts())

print("\nClicks:")
print(camp_wa_002["clicked"].value_counts())

print("\nSent timestamp range:")
print(camp_wa_002["sent_ts"].min())
print(camp_wa_002["sent_ts"].max())

print("\nUnique captains:")
print(camp_wa_002["captain_id"].nunique())

print("\nRows per captain:")
print(camp_wa_002["captain_id"].value_counts().value_counts().sort_index())

CAMP_WA_002 shape: (8673, 6)

Channels:
channel
whatsapp    8673
Name: count, dtype: int64

Delivery:
delivered
1    8058
0     615
Name: count, dtype: int64

Clicks:
clicked
0    5104
1    3569
Name: count, dtype: int64

Sent timestamp range:
2026-01-02 12:07:52.213424314
2026-06-30 23:37:42.004031897

Unique captains:
8673

Rows per captain:
count
1    8673
Name: count, dtype: int64


In [109]:
campaign_timing = camp_wa_002.merge(
    captain_base[
        [
            "captain_id",
            "signup_ts",
            "decision_ts",
            "final_status",
            "eligible_for_funnel"
        ]
    ],
    on="captain_id",
    how="left",
    validate="one_to_one"
)

campaign_timing["hours_after_signup"] = (
    campaign_timing["sent_ts"] - campaign_timing["signup_ts"]
).dt.total_seconds() / 3600

campaign_timing["hours_before_decision"] = (
    campaign_timing["decision_ts"] - campaign_timing["sent_ts"]
).dt.total_seconds() / 3600

print("Campaign rows after join:", len(campaign_timing))

print("\nSent before signup:")
print((campaign_timing["hours_after_signup"] < 0).sum())

print("\nSent after signup:")
print((campaign_timing["hours_after_signup"] >= 0).sum())

print("\nSent before approval decision:")
print((campaign_timing["hours_before_decision"] > 0).sum())

print("\nSent after approval decision:")
print((campaign_timing["hours_before_decision"] <= 0).sum())

print("\nHours after signup:")
print(campaign_timing["hours_after_signup"].describe())

print("\nHours before decision:")
print(campaign_timing["hours_before_decision"].describe())

Campaign rows after join: 8673

Sent before signup:
0

Sent after signup:
8673

Sent before approval decision:
2727

Sent after approval decision:
0

Hours after signup:
count    8673.000000
mean       56.721898
std        20.757396
min         9.404725
25%        41.855489
50%        53.531545
75%        68.325829
max       184.939012
Name: hours_after_signup, dtype: float64

Hours before decision:
count    2727.000000
mean       97.942663
std        31.566859
min        11.134250
25%        75.210191
50%        95.157311
75%       117.956479
max       238.455229
Name: hours_before_decision, dtype: float64


In [111]:
pre_decision_campaign = campaign_timing[
    campaign_timing["hours_before_decision"] > 0
].copy()

pre_decision_campaign = pre_decision_campaign.merge(
    captain_base[
        [
            "captain_id",
            "city",
            "vehicle_type",
            "acquisition_channel",
            "device_tier",
            "app_language",
            "age_band"
        ]
    ],
    on="captain_id",
    how="left",
    validate="one_to_one"
)

print("Pre-decision campaign recipients:", len(pre_decision_campaign))

for col in [
    "city",
    "vehicle_type",
    "acquisition_channel",
    "device_tier",
    "app_language",
    "age_band"
]:
    print(f"\n--- {col} ---")
    print(
        pre_decision_campaign[col]
        .value_counts(normalize=True)
        .mul(100)
        .round(2)
    )

Pre-decision campaign recipients: 2727

--- city ---
city
Hyderabad    30.66
Delhi        27.61
Bangalore    21.05
Pune         20.68
Name: proportion, dtype: float64

--- vehicle_type ---
vehicle_type
Auto         41.03
Cab          34.25
ERickshaw    24.72
Name: proportion, dtype: float64

--- acquisition_channel ---
acquisition_channel
fos_field         37.40
organic_app       28.75
referral          20.46
paid_digital       7.77
gc_telecalling     5.61
Name: proportion, dtype: float64

--- device_tier ---
device_tier
mid     43.31
low     36.60
high    20.10
Name: proportion, dtype: float64

--- app_language ---
app_language
te    20.68
kn    20.61
hi    20.50
en    19.51
mr    18.70
Name: proportion, dtype: float64

--- age_band ---
age_band
25-34    40.89
35-44    30.03
18-24    17.57
45+      11.51
Name: proportion, dtype: float64


In [112]:
campaigns_pre_decision = campaign_timing[
    campaign_timing["hours_before_decision"] > 0
].copy()

print(
    campaigns_pre_decision["campaign_id"]
    .value_counts()
)

print("\nShare of pre-decision campaign exposures:")
print(
    campaigns_pre_decision["campaign_id"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

campaign_id
CAMP_WA_002    2727
Name: count, dtype: int64

Share of pre-decision campaign exposures:
campaign_id
CAMP_WA_002    100.0
Name: proportion, dtype: float64


In [113]:
print("Hours after signup — pre-decision recipients:")
print(
    pre_decision_campaign["hours_after_signup"]
    .describe(
        percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]
    )
)

print("\nRounded hours after signup:")
print(
    pre_decision_campaign["hours_after_signup"]
    .round()
    .value_counts()
    .sort_index()
    .head(30)
)

Hours after signup — pre-decision recipients:
count    2727.000000
mean       56.679044
std        20.710358
min        13.700235
10%        32.897922
25%        41.989234
50%        53.631199
75%        68.377675
90%        83.894296
max       184.939012
Name: hours_after_signup, dtype: float64

Rounded hours after signup:
hours_after_signup
14.0     1
15.0     1
16.0     4
17.0     1
18.0     3
19.0     6
20.0     2
21.0     7
22.0    13
23.0    13
24.0    21
25.0    17
26.0    17
27.0    18
28.0    15
29.0    23
30.0    29
31.0    23
32.0    40
33.0    42
34.0    35
35.0    41
36.0    59
37.0    42
38.0    40
39.0    38
40.0    49
41.0    65
42.0    36
43.0    58
Name: count, dtype: int64


In [114]:
print("\nDays after signup:")
print(
    (
        pre_decision_campaign["hours_after_signup"] / 24
    ).describe(
        percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]
    )
)


Days after signup:
count    2727.000000
mean        2.361627
std         0.862932
min         0.570843
10%         1.370747
25%         1.749551
50%         2.234633
75%         2.849070
90%         3.495596
max         7.705792
Name: hours_after_signup, dtype: float64


In [115]:
# Captains whose approval decision happened at least 8 days after signup
# are eligible for a clean 8-day pre-decision exposure comparison.

comparison = captain_base[
    captain_base["decision_ts"].notna()
    & (
        (captain_base["decision_ts"] - captain_base["signup_ts"])
        .dt.total_seconds() / 86400
        >= 8
    )
].copy()

print("Comparison population:", len(comparison))

Comparison population: 675


In [116]:
campaign_8d = camp_wa_002.merge(
    captain_base[
        ["captain_id", "signup_ts", "decision_ts", "final_status"]
    ],
    on="captain_id",
    how="left",
    validate="one_to_one"
)

campaign_8d["days_after_signup"] = (
    campaign_8d["sent_ts"] - campaign_8d["signup_ts"]
).dt.total_seconds() / 86400

exposed_ids = set(
    campaign_8d.loc[
        campaign_8d["days_after_signup"].between(0, 8, inclusive="both"),
        "captain_id"
    ]
)

comparison["campaign_exposed"] = comparison["captain_id"].isin(exposed_ids)

print("\nExposure:")
print(comparison["campaign_exposed"].value_counts())


Exposure:
campaign_exposed
True     395
False    280
Name: count, dtype: int64


In [117]:
approval_comparison = (
    comparison
    .groupby("campaign_exposed")["final_status"]
    .apply(lambda x: (x == "approved").mean() * 100)
)

print("\nApproval rate:")
print(approval_comparison.round(2))


Approval rate:
campaign_exposed
False    91.43
True     90.63
Name: final_status, dtype: float64


In [118]:
print("\nCounts:")
print(
    comparison.groupby("campaign_exposed")["final_status"]
    .agg(
        captains="size",
        approved=lambda x: (x == "approved").sum()
    )
)


Counts:
                  captains  approved
campaign_exposed                    
False                  280       256
True                   395       358


In [119]:
import math

treated = comparison[comparison["campaign_exposed"]]
control = comparison[~comparison["campaign_exposed"]]

p_treated = (treated["final_status"] == "approved").mean()
p_control = (control["final_status"] == "approved").mean()

n_treated = len(treated)
n_control = len(control)

diff = p_treated - p_control

se = math.sqrt(
    (p_treated * (1 - p_treated) / n_treated)
    +
    (p_control * (1 - p_control) / n_control)
)

ci_low = diff - 1.96 * se
ci_high = diff + 1.96 * se

print("Observed difference:", round(diff * 100, 2), "pp")
print(
    "95% CI:",
    round(ci_low * 100, 2),
    "pp to",
    round(ci_high * 100, 2),
    "pp"
)

Observed difference: -0.8 pp
95% CI: -5.16 pp to 3.56 pp


In [120]:
for col in [
    "city",
    "vehicle_type",
    "acquisition_channel",
    "device_tier",
    "app_language",
    "age_band"
]:
    print(f"\n{'='*50}")
    print(col)

    rates = pd.crosstab(
        comparison[col],
        comparison["campaign_exposed"],
        normalize="columns"
    ).mul(100).round(2)

    print(rates)


city
campaign_exposed  False  True 
city                          
Bangalore         24.29  23.29
Delhi             23.21  23.04
Hyderabad         33.57  30.38
Pune              18.93  23.29

vehicle_type
campaign_exposed  False  True 
vehicle_type                  
Auto              50.36  50.63
Cab               39.64  38.99
ERickshaw         10.00  10.38

acquisition_channel
campaign_exposed     False  True 
acquisition_channel              
fos_field            34.64  35.70
gc_telecalling        6.07   4.05
organic_app          31.07  31.65
paid_digital          9.29   8.10
referral             18.93  20.51

device_tier
campaign_exposed  False  True 
device_tier                   
high              15.00  15.95
low               40.36  44.05
mid               44.64  40.00

app_language
campaign_exposed  False  True 
app_language                  
en                24.29  20.00
hi                19.29  16.96
kn                21.79  24.05
mr                16.07  19.75
te          

In [121]:
stratified = (
    comparison
    .groupby(
        ["city", "acquisition_channel", "device_tier", "campaign_exposed"]
    )["final_status"]
    .agg(
        captains="size",
        approved=lambda x: (x == "approved").sum()
    )
    .reset_index()
)

stratified["approval_rate"] = (
    stratified["approved"] / stratified["captains"]
)

print(stratified.head(20))
print("\nNumber of strata:", len(stratified))

         city acquisition_channel device_tier  campaign_exposed  captains  \
0   Bangalore           fos_field        high             False         4   
1   Bangalore           fos_field        high              True         5   
2   Bangalore           fos_field         low             False        11   
3   Bangalore           fos_field         low              True        16   
4   Bangalore           fos_field         mid             False         9   
5   Bangalore           fos_field         mid              True        17   
6   Bangalore      gc_telecalling        high              True         1   
7   Bangalore      gc_telecalling         low             False         3   
8   Bangalore      gc_telecalling         low              True         1   
9   Bangalore      gc_telecalling         mid             False         2   
10  Bangalore         organic_app        high             False         5   
11  Bangalore         organic_app        high              True        12   

In [122]:
overlap = (
    comparison
    .groupby(
        ["city", "acquisition_channel", "device_tier"]
    )["campaign_exposed"]
    .nunique()
)

print("Strata with both exposed and unexposed:", (overlap == 2).sum())
print("Total strata:", len(overlap))
print("Strata with only one group:", (overlap == 1).sum())

Strata with both exposed and unexposed: 50
Total strata: 59
Strata with only one group: 9


In [123]:
overlap_strata = overlap[overlap == 2].index

matched_population = comparison.set_index(
    ["city", "acquisition_channel", "device_tier"]
).loc[overlap_strata].reset_index()

print("Captains in overlapping strata:", len(matched_population))

standardized = (
    matched_population
    .groupby("campaign_exposed")
    .apply(
        lambda g: pd.Series({
            "captains": len(g),
            "approved": (g["final_status"] == "approved").sum(),
            "approval_rate": (
                (g["final_status"] == "approved").mean()
            )
        }),
        include_groups=False
    )
)

print(standardized)

Captains in overlapping strata: 652
                  captains  approved  approval_rate
campaign_exposed                                   
False                276.0     252.0       0.913043
True                 376.0     344.0       0.914894


In [124]:
std_diff = (
    standardized.loc[True, "approval_rate"]
    - standardized.loc[False, "approval_rate"]
)

print(
    "Standardized difference:",
    round(std_diff * 100, 2),
    "pp"
)

Standardized difference: 0.19 pp


In [125]:
print("Total CAMP_WA_002 recipients:", len(camp_wa_002))

print(
    "Pre-decision recipients:",
    len(pre_decision_campaign)
)

print(
    "Pre-decision share:",
    round(
        len(pre_decision_campaign) / len(camp_wa_002) * 100,
        2
    ),
    "%"
)

print("\nTotal captains:", len(captain_base))

print(
    "CAMP_WA_002 recipient share of all captains:",
    round(
        len(camp_wa_002) / len(captain_base) * 100,
        2
    ),
    "%"
)

Total CAMP_WA_002 recipients: 8673
Pre-decision recipients: 2727
Pre-decision share: 31.44 %

Total captains: 25000
CAMP_WA_002 recipient share of all captains: 34.69 %


In [126]:
comparison["days_to_decision"] = (
    comparison["decision_ts"] - comparison["signup_ts"]
).dt.total_seconds() / 86400

print(
    comparison
    .groupby("campaign_exposed")["days_to_decision"]
    .describe()
    .round(2)
)

                  count  mean   std  min   25%   50%   75%    max
campaign_exposed                                                 
False             280.0  9.10  0.94  8.0  8.37  8.88  9.59  12.79
True              395.0  9.04  0.87  8.0  8.35  8.84  9.49  14.11


In [127]:
print("\nMedian days to decision:")
print(
    comparison
    .groupby("campaign_exposed")["days_to_decision"]
    .median()
    .round(2)
)

print("\nFinal status distribution:")
print(
    pd.crosstab(
        comparison["campaign_exposed"],
        comparison["final_status"],
        normalize="index"
    ).mul(100).round(2)
)


Median days to decision:
campaign_exposed
False    8.88
True     8.84
Name: days_to_decision, dtype: float64

Final status distribution:
final_status      approved  rejected
campaign_exposed                    
False                91.43      8.57
True                 90.63      9.37


CAMP_WA_002 evaluation

CAMP_WA_002 was sent to 8,673 captains, but only 2,727
(31.4%) received it before their approval decision and could
plausibly be affected by the campaign.

For a cleaner comparison, we restricted analysis to 675 captains
whose signup-to-decision time was at least 8 days, giving both
exposed and unexposed captains sufficient opportunity for campaign
exposure.

Among this cohort:
- Exposed: 395; approved = 358 (90.63%)
- Unexposed: 280; approved = 256 (91.43%)
- Raw difference: -0.80 percentage points
- 95% CI: -5.16 to +3.56 percentage points

After restricting to the 50 city × acquisition_channel × device_tier
strata with both exposed and unexposed captains:
- Exposed approval rate: 91.49%
- Unexposed approval rate: 91.30%
- Standardized difference: +0.19 percentage points

Decision timing was also similar:
- Mean days to decision: 9.04 exposed vs 9.10 unexposed
- Median: 8.84 vs 8.88 days

Interpretation:
The available observational data shows no detectable approval lift
from CAMP_WA_002. The adjusted estimate is effectively zero.
Because campaign assignment was not randomized, this should not be
interpreted as proof of zero causal effect.

Recommendation:
Do not scale CAMP_WA_002 5× based on the current evidence.
Instead, run a randomized holdout experiment among eligible
pre-decision captains and measure approval lift as the primary
outcome before scaling.